In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Define the directory containing .out files
dir_path = "/data/antismash/gbk_protocluster_parse"

In [ ]:
# Initialize a list to store individual summary DataFrames
all_summaries = []

# Key mapping dictionary for assembler names
key_map = {
    "megahit": "MEGAHIT",
    "spades": "metaSPAdes",
    "penguin": "penguiN",
    "carpedeam.configSafe0": "CarpeDeam\n(Safe)",
    "carpedeam.configUnsafe0": "CarpeDeam\n(Unsafe)"
}

In [ ]:
# Iterate over all .out files in the directory
for filename in os.listdir(dir_path):
    if filename.endswith(".out"):
        file_path = os.path.join(dir_path, filename)
        # Read the CSV file; adjust the separator if needed
        df = pd.read_csv(file_path, sep=",")
        
        # Compute a summary by grouping by 'product'
        summary = df.groupby("product").agg(
            total_alignment_length=("gbk_end", lambda x: (x - df.loc[x.index, "gbk_start"]).sum()),
            count=("product", "count")
        ).reset_index()
        
        # If the summary is empty, add a dummy row with zero counts
        if summary.empty:
            summary = pd.DataFrame({
                "product": ["No Product"],
                "total_alignment_length": [0],
                "count": [0]
            })
        
        # Extract sample ID (everything before ".raw-raw")
        sample_id = filename.split(".raw-raw")[0]
        
        # Determine the assembler name based on key_map
        assembler = None
        for key, value in key_map.items():
            if key in filename:
                assembler = value
                break
        if assembler is None:
            assembler = filename  # Fallback if no key matches
        
        # Add sample_id and assembler columns to the summary
        summary["sample_id"] = sample_id
        summary["assembler"] = assembler
        
        # Append the DataFrame to our list
        all_summaries.append(summary)

# Combine all summaries into a single big DataFrame
big_df = pd.concat(all_summaries, ignore_index=True)
print(big_df.head())

# For each unique sample (dataset), generate a heatmap:
unique_samples = big_df["sample_id"].unique()
for sample in unique_samples:
    # Filter for the current sample
    sample_df = big_df[big_df["sample_id"] == sample]
    
    pivot_df = sample_df.pivot_table(
        index="assembler",
        columns="product",
        values="count",
        aggfunc="sum",
        fill_value=0
    )
    
    # Remove the "No Product" column if it exists
    if "No Product" in pivot_df.columns:
        pivot_df = pivot_df.drop(columns="No Product")
    
    # Plot the heatmap using seaborn with the "coolwarm" palette
    plt.figure(figsize=(7, 4))
    sns.heatmap(pivot_df, annot=True, fmt="d", cmap="coolwarm")
    plt.title(f"BGC Protoclusters in {sample}")
    plt.xlabel("BGC type", fontsize=12)
    plt.ylabel("Assembler",fontsize=12)
    plt.tight_layout()
    plt.show()